# Pipeline de ML passo a passo

Este notebook reorganiza sua pipeline original em células sequenciais, sem classe e sem funções customizadas para o fluxo principal.
A ideia é executar **de cima para baixo**, uma célula por vez.

In [1]:
import pandas as pd
import numpy as np
import warnings
import unicodedata

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

## 1) Definir o caminho do arquivo CSV

Ajuste o nome do arquivo se necessário.

In [2]:
file_path = "Dados Históricos - Ibovespa 2006.csv"
train_start_date = pd.to_datetime("2020-01-01")
target_threshold = 0.003
test_size = 30
cutoff_proba = 0.45

## 2) Ler os dados brutos

In [3]:
df_raw = pd.read_csv(file_path)
df_raw.head()

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%
0,02.03.2026,189.307,188.786,190.110,186.638,"9,09B","0,28%"
1,27.02.2026,188.787,191.005,191.005,188.478,"11,00B","-1,16%"
2,26.02.2026,191.005,191.248,191.978,188.977,"8,80B","-0,13%"
3,25.02.2026,191.247,191.491,192.624,190.419,"9,13B","-0,13%"
4,24.02.2026,191.490,188.854,191.781,188.854,"11,04B","1,40%"


## 3) Limpar os nomes das colunas

In [4]:
def strip_accents(s):
    return ''.join(
        ch for ch in unicodedata.normalize('NFKD', s)
        if not unicodedata.combining(ch)
    )

def clean_col(c):
    return strip_accents(str(c)).replace('.', '').replace('%', '').lower().strip()

cols = {clean_col(c): c for c in df_raw.columns}
cols

{'data': 'Data',
 'ultimo': 'Último',
 'abertura': 'Abertura',
 'maxima': 'Máxima',
 'minima': 'Mínima',
 'vol': 'Vol.',
 'var': 'Var%'}

## 4) Renomear as colunas principais para um padrão único

In [5]:
rename = {}

for k, v in cols.items():
    if "data" in k:
        rename[v] = "Date"
    elif "ultimo" in k or "fechamento" in k:
        rename[v] = "Close"
    elif "abert" in k:
        rename[v] = "Open"
    elif "max" in k:
        rename[v] = "High"
    elif "min" in k:
        rename[v] = "Low"
    elif "vol" in k:
        rename[v] = "Volume"

rename

{'Data': 'Date',
 'Último': 'Close',
 'Abertura': 'Open',
 'Máxima': 'High',
 'Mínima': 'Low',
 'Vol.': 'Volume'}

## 5) Padronizar os formatos numéricos

In [6]:
df = df_raw.rename(columns=rename).copy()

def coerce_pt_number(s):
    if pd.isna(s):
        return np.nan
    s = str(s).strip().replace('.', '').replace(',', '.')
    try:
        return float(s)
    except:
        return np.nan

def parse_pt_volume(v):
    if pd.isna(v):
        return np.nan

    s = str(v).strip()
    mult = 1

    if s.endswith(("M", "m")):
        mult, s = 1e6, s[:-1]
    elif s.endswith(("K", "k")):
        mult, s = 1e3, s[:-1]
    elif s.endswith(("B", "b")):
        mult, s = 1e9, s[:-1]

    try:
        return float(s.replace('.', '').replace(',', '.')) * mult
    except:
        return np.nan

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for col in ["Close", "Open", "High", "Low"]:
    if col in df.columns:
        df[col] = df[col].apply(coerce_pt_number)

if "Volume" in df.columns:
    df["Volume"] = df["Volume"].apply(parse_pt_volume)

df = df.dropna(subset=["Date"]).sort_values("Date").set_index("Date")
df.head()

,Close,Open,High,Low,Volume,Var%
Date,,,,,,
2006-03-02,39126.0,39182.0,39367.0,38989.0,86850000.0,"-0,13%"
2006-03-03,3924.0,39129.0,39356.0,38716.0,96080000.0,"0,29%"
2006-03-06,38354.0,39241.0,39395.0,38348.0,125160000.0,"-2,26%"
2006-03-07,37423.0,3835.0,3835.0,3713.0,150700000.0,"-2,43%"
2006-03-08,37289.0,37421.0,37421.0,36446.0,139700000.0,"-0,36%"


## 6) Conferir tipos e valores ausentes

In [7]:
print(df.dtypes)
print()
print(df.isna().sum())

Close     float64
Open      float64
High      float64
Low       float64
Volume    float64
Var%       object
dtype: object

Close     0
Open      0
High      0
Low       0
Volume    1
Var%      0
dtype: int64


## 7) Criar a variável alvo (target)

Aqui o alvo é:
- 1 = o retorno do próximo dia foi maior que o limiar
- 0 = caso contrário

In [8]:
df["Next_Close"] = df["Close"].shift(-1)
df["Retorno_Pct"] = df["Close"].pct_change().shift(-1)
df["Target"] = (df["Retorno_Pct"] > target_threshold).astype(int)

df[["Close", "Next_Close", "Retorno_Pct", "Target"]].head(10)

,Close,Next_Close,Retorno_Pct,Target
Date,,,,
2006-03-02,39126.0,3924.0,-0.899709,0
2006-03-03,3924.0,38354.0,8.774210,1
2006-03-06,38354.0,37423.0,-0.024274,0
2006-03-07,37423.0,37289.0,-0.003581,0
2006-03-08,37289.0,36312.0,-0.026201,0
2006-03-09,36312.0,36891.0,0.015945,1
2006-03-10,36891.0,36793.0,-0.002656,0
2006-03-13,36793.0,37541.0,0.020330,1
2006-03-14,37541.0,38244.0,0.018726,1


## 8) Criar variáveis de retorno e lags

In [9]:
df["Ret"] = df["Close"].pct_change() * 100

for k in [1, 2, 3, 5, 10]:
    df[f"Ret_lag_{k}"] = df["Ret"].shift(k)

df.filter(like="Ret").head(10)

,Retorno_Pct,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10
Date,,,,,,,
2006-03-02,-0.899709,NaN,NaN,NaN,NaN,NaN,NaN
2006-03-03,8.774210,-89.970863,NaN,NaN,NaN,NaN,NaN
2006-03-06,-0.024274,877.420999,-89.970863,NaN,NaN,NaN,NaN
2006-03-07,-0.003581,-2.427387,877.420999,-89.970863,NaN,NaN,NaN
2006-03-08,-0.026201,-0.358069,-2.427387,877.420999,-89.970863,NaN,NaN
2006-03-09,0.015945,-2.620076,-0.358069,-2.427387,877.420999,NaN,NaN
2006-03-10,-0.002656,1.594514,-2.620076,-0.358069,-2.427387,-89.970863,NaN
2006-03-13,0.020330,-0.265647,1.594514,-2.620076,-0.358069,877.420999,NaN
2006-03-14,0.018726,2.032995,-0.265647,1.594514,-2.620076,-2.427387,NaN


## 9) Criar médias móveis e distâncias

In [10]:
df["SMA_5"] = df["Close"].rolling(5).mean()
df["SMA_10"] = df["Close"].rolling(10).mean()
df["SMA_20"] = df["Close"].rolling(20).mean()

df["EMA_10"] = df["Close"].ewm(span=10).mean()
df["EMA_20"] = df["Close"].ewm(span=20).mean()

df["Dist_SMA5"] = (df["Close"] - df["SMA_5"]) / df["SMA_5"]
df["Dist_SMA20"] = (df["Close"] - df["SMA_20"]) / df["SMA_20"]

df[["SMA_5", "SMA_10", "SMA_20", "EMA_10", "EMA_20", "Dist_SMA5", "Dist_SMA20"]].tail()

,SMA_5,SMA_10,SMA_20,EMA_10,EMA_20,Dist_SMA5,Dist_SMA20
Date,,,,,,,
2026-02-24,154617.2,170918.5,168480.55,152099.803093,158285.222646,-0.876152,-0.886343
2026-02-25,155663.4,171419.1,169106.85,159217.475258,161424.439537,0.228593,0.130924
2026-02-26,156157.6,171926.7,169561.15,164997.025211,164241.635771,0.223155,0.126467
2026-02-27,155808.2,171835.5,169765.95,169322.475173,166579.289507,0.211663,0.112043
2026-03-02,155899.0,171989.6,170074.60,172956.025141,168743.833364,0.214293,0.113082


## 10) Criar variáveis de momentum e volatilidade

In [11]:
df["Momentum_3"] = df["Close"] - df["Close"].shift(3)
df["Momentum_7"] = df["Close"] - df["Close"].shift(7)
df["Momentum_14"] = df["Close"] - df["Close"].shift(14)

df["Vol_5"] = df["Close"].pct_change().rolling(5).std()
df["Vol_10"] = df["Close"].pct_change().rolling(10).std()
df["Vol_20"] = df["Close"].pct_change().rolling(20).std()

df[["Momentum_3", "Momentum_7", "Momentum_14", "Vol_5", "Vol_10", "Vol_20"]].tail()

,Momentum_3,Momentum_7,Momentum_14,Vol_5,Vol_10,Vol_20
Date,,,,,,
2026-02-24,-169385.0,-170550.0,-163644.0,0.403417,2.947634,2.091843
2026-02-25,713.0,3481.0,5573.0,4.136561,2.887024,2.839645
2026-02-26,2152.0,4541.0,9297.0,4.138004,2.887011,2.839933
2026-02-27,169638.0,2771.0,6660.0,4.140173,2.887998,2.840340
2026-03-02,-1940.0,773.0,171012.0,4.139042,2.887594,2.840170


## 11) Criar RSI

In [12]:
delta = df["Close"].diff()
up = delta.clip(lower=0).rolling(14).mean()
down = (-delta.clip(upper=0)).rolling(14).mean()
rs = up / down
df["RSI_14"] = 100 - (100 / (1 + rs))

df[["Close", "RSI_14"]].tail()

,Close,RSI_14
Date,,
2026-02-24,19149.0,34.346638
2026-02-25,191247.0,50.402715
2026-02-26,191005.0,50.675453
2026-02-27,188787.0,50.482606
2026-03-02,189307.0,66.234536


## 12) Criar MACD

In [13]:
ema_fast = df["Close"].ewm(span=12).mean()
ema_slow = df["Close"].ewm(span=26).mean()
df["MACD"] = ema_fast - ema_slow

df[["Close", "MACD"]].tail()

,Close,MACD
Date,,
2026-02-24,19149.0,-2620.519112
2026-02-25,191247.0,463.135922
2026-02-26,191005.0,2854.522220
2026-02-27,188787.0,4518.650833
2026-03-02,189307.0,5812.442187


## 13) Criar Bollinger %B

In [14]:
bb_mid = df["Close"].rolling(20).mean()
bb_std = df["Close"].rolling(20).std()
bb_up = bb_mid + 2 * bb_std
bb_low = bb_mid - 2 * bb_std

df["BB_pctB"] = (df["Close"] - bb_low) / (bb_up - bb_low)

df[["Close", "BB_pctB"]].tail()

,Close,BB_pctB
Date,,
2026-02-24,19149.0,-0.227587
2026-02-25,191247.0,0.607439
2026-02-26,191005.0,0.603740
2026-02-27,188787.0,0.591893
2026-03-02,189307.0,0.592730


## 14) Criar Stochastic %K

In [15]:
lowest_low = df["Low"].rolling(14).min()
highest_high = df["High"].rolling(14).max()

df["StochK"] = 100 * (df["Close"] - lowest_low) / (highest_high - lowest_low)

df[["Close", "StochK"]].tail()

,Close,StochK
Date,,
2026-02-24,19149.0,9.099908
2026-02-25,191247.0,99.278139
2026-02-26,191005.0,99.151276
2026-02-27,188787.0,97.988540
2026-03-02,189307.0,98.261139


## 15) Criar variáveis extras de tendência, candle e calendário

In [16]:
df["Trend"] = (df["SMA_5"] > df["SMA_20"]).astype(int)
df["Range"] = df["High"] - df["Low"]
df["Body"] = df["Close"] - df["Open"]
df["dow"] = df.index.dayofweek
df["month"] = df.index.month

df[["Trend", "Range", "Body", "dow", "month"]].tail()

,Trend,Range,Body,dow,month
Date,,,,,
2026-02-24,0,2927.0,-169705.0,1,2
2026-02-25,0,2205.0,-244.0,2,2
2026-02-26,0,3001.0,-243.0,3,2
2026-02-27,0,2527.0,-2218.0,4,2
2026-03-02,0,-167627.0,521.0,0,3


## 16) Remover linhas com valores ausentes gerados pelos cálculos

In [17]:
df_model = df.dropna().copy()
df_model.shape

(4932, 37)

## 17) Separar X e y

In [18]:
X = df_model.drop(
    columns=["Open", "High", "Low", "Close", "Next_Close", "Target", "Retorno_Pct"],
    errors="ignore"
)

X = X.select_dtypes(exclude="object")
y = df_model["Target"]

print(X.shape)
print(y.shape)
X.head()

(4932, 29)
(4932,)


,Volume,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10,SMA_5,SMA_10,SMA_20,...,Vol_20,RSI_14,MACD,BB_pctB,StochK,Trend,Range,Body,dow,month
Date,,,,,,,,,,,,,,,,,,,,,
2006-03-30,114600000.0,0.760162,2.208167,-2.547754,0.170317,-0.996011,-0.227487,37433.8,37614.5,35853.65,...,1.982958,57.696317,467.851732,0.563810,62.933827,1,620.0,284.0,3,3
2006-03-31,118950000.0,0.463245,0.760162,2.208167,-2.547754,0.274857,-0.283041,37508.8,37604.8,37555.05,...,1.962129,59.934853,467.513822,0.678154,71.031930,0,397.0,169.0,4,3
2006-04-03,95110000.0,2.015704,0.463245,0.760162,2.208167,0.170317,0.407369,37724.0,37656.1,37573.20,...,0.015662,60.051282,513.681854,0.985147,93.822038,1,899.0,765.0,0,4
2006-04-04,127330000.0,0.219542,2.015704,0.463245,0.760162,-2.547754,-2.109727,38148.0,37796.5,37642.15,...,0.014527,55.332569,549.319931,0.947059,88.186356,1,480.0,84.0,1,4
2006-04-05,158410000.0,0.646874,0.219542,2.015704,0.463245,2.208167,1.211295,38460.2,37916.7,37730.35,...,0.014501,58.302446,588.020241,0.962706,98.627288,1,579.0,250.0,2,4


## 18) Fazer o corte temporal entre treino e teste

In [19]:
X_train = X.iloc[:-test_size].copy()
y_train = y.iloc[:-test_size].copy()

X_test = X.iloc[-test_size:].copy()
y_test = y.iloc[-test_size:].copy()

X_train = X_train[X_train.index >= train_start_date]
y_train = y_train[y_train.index >= train_start_date]

print("Treino:", X_train.shape, y_train.shape)
print("Teste:", X_test.shape, y_test.shape)

Treino: (1504, 29) (1504,)
Teste: (30, 29) (30,)


## 19) Padronizar os dados

In [20]:
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    index=X_train.index,
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    index=X_test.index,
    columns=X_test.columns
)

X_train_scaled.head()

,Volume,Ret,Ret_lag_1,Ret_lag_2,Ret_lag_3,Ret_lag_5,Ret_lag_10,SMA_5,SMA_10,SMA_20,...,Vol_20,RSI_14,MACD,BB_pctB,StochK,Trend,Range,Body,dow,month
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-02,-0.436173,-0.163241,-0.166602,-0.165829,-0.164067,-0.165319,9.656974,0.526239,0.537054,0.096617,...,2.037690,0.090956,1.175488,0.432362,0.224183,0.92563,0.036732,0.070697,0.710760,-1.592742
2020-01-03,-0.435706,-0.166570,-0.163239,-0.166025,-0.165830,-0.164652,-0.166149,0.544356,0.570865,0.125374,...,2.037711,0.081600,1.187881,0.379836,0.220787,0.92563,0.005345,-0.009047,1.419166,-1.592742
2020-01-06,-0.435779,-0.166544,-0.166569,-0.162662,-0.166026,-0.164130,-0.164925,0.541163,0.595664,0.151324,...,2.037721,0.063276,1.169670,0.331143,0.218192,0.92563,0.005068,-0.008456,-1.414455,-1.592742
2020-01-07,-0.436260,-0.166014,-0.166542,-0.165992,-0.162663,-0.165893,-0.164069,0.542421,0.609320,0.172161,...,2.037742,3.477798,1.138052,0.309624,0.217515,0.92563,-0.001900,0.004610,-0.706050,-1.592742
2020-01-08,-0.435963,-0.166188,-0.166012,-0.165965,-0.165992,-0.166089,-0.164855,0.548335,0.615813,0.190586,...,2.037752,1.049419,1.092925,0.282092,0.216216,0.92563,0.009415,0.000177,0.002355,-1.592742


## 20) Treinar Random Forest

In [21]:
rf = RandomForestClassifier(
    n_estimators=1500,
    max_depth=12,
    min_samples_split=4,
    min_samples_leaf=1,
    max_features=0.7,
    max_samples=0.8,
    bootstrap=True,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

proba_rf = rf.predict_proba(X_test_scaled)[:, 1]
pred_rf = (proba_rf >= cutoff_proba).astype(int)

acc_rf = accuracy_score(y_test, pred_rf)
hits_rf = (y_test == pred_rf).sum()

print(f"RandomForest: {acc_rf * 100:.2f}% | {hits_rf}/{len(y_test)}")

RandomForest: 73.33% | 22/30


## 21) Treinar XGBoost

In [22]:
xgb = XGBClassifier(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    reg_lambda=1,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train_scaled, y_train)

proba_xgb = xgb.predict_proba(X_test_scaled)[:, 1]
pred_xgb = (proba_xgb >= cutoff_proba).astype(int)

acc_xgb = accuracy_score(y_test, pred_xgb)
hits_xgb = (y_test == pred_xgb).sum()

print(f"XGBoost: {acc_xgb * 100:.2f}% | {hits_xgb}/{len(y_test)}")

XGBoost: 80.00% | 24/30


## 22) Treinar SVM

In [23]:
svm = SVC(
    kernel="rbf",
    C=20,
    gamma=0.05,
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm.fit(X_train_scaled, y_train)

proba_svm = svm.predict_proba(X_test_scaled)[:, 1]
pred_svm = (proba_svm >= cutoff_proba).astype(int)

acc_svm = accuracy_score(y_test, pred_svm)
hits_svm = (y_test == pred_svm).sum()

print(f"SVM: {acc_svm * 100:.2f}% | {hits_svm}/{len(y_test)}")

SVM: 63.33% | 19/30


## 23) Comparar os modelos

In [24]:
results = {
    "RandomForest": (acc_rf * 100, hits_rf, len(y_test), rf),
    "XGBoost": (acc_xgb * 100, hits_xgb, len(y_test), xgb),
    "SVM": (acc_svm * 100, hits_svm, len(y_test), svm),
}

for name, (acc, hits, total, _) in results.items():
    print(f"{name}: {acc:.2f}% | {hits}/{total}")

RandomForest: 73.33% | 22/30
XGBoost: 80.00% | 24/30
SVM: 63.33% | 19/30


## 24) Selecionar o melhor modelo

In [25]:
best_name = max(results.items(), key=lambda x: x[1][0])[0]
best_model = results[best_name][3]

print("Melhor modelo:", best_name)

Melhor modelo: XGBoost


## 25) Gerar o classification report do melhor modelo

In [26]:
proba_best = best_model.predict_proba(X_test_scaled)[:, 1]
pred_best = (proba_best >= cutoff_proba).astype(int)

print(classification_report(y_test, pred_best))

              precision    recall  f1-score   support

           0       0.75      0.94      0.83        16
           1       0.90      0.64      0.75        14

    accuracy                           0.80        30
   macro avg       0.82      0.79      0.79        30
weighted avg       0.82      0.80      0.79        30



## 26) Conferir as previsões finais

In [27]:
df_resultado = pd.DataFrame({
    "y_real": y_test,
    "proba": proba_best,
    "y_pred": pred_best
}, index=y_test.index)

df_resultado.tail(30)

,y_real,proba,y_pred
Date,,,
2026-01-15,0,0.336276,0
2026-01-16,1,0.987164,1
2026-01-19,1,0.814299,1
2026-01-20,1,0.245709,0
2026-01-21,1,0.632928,1
2026-01-22,1,0.346489,0
2026-01-23,0,0.270085,0
2026-01-26,1,0.718176,1
2026-01-27,1,0.570433,1
